# Abdul Model Jupyter Runner

Dieses Notebook initialisiert den aktuellen Branch-Stand robust, laedt einen frischen 2%-Testdatensatz fuer Abdul und startet danach den Wrapper `run_abdul_model.py`, ohne Abduls Code unter `model-training/` zu veraendern.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

repo_name = 'Hackathon-TUM'
branch_name = 'feature/jupyter-integration'

cwd = Path.cwd().resolve()
while cwd.name == repo_name:
    cwd = cwd.parent
os.chdir(cwd)

repo_dir = Path.cwd() / repo_name
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Say43/Hackathon-TUM.git', repo_name], check=True)

subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', branch_name], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', branch_name], check=True)

os.chdir(repo_dir)
print('Working directory:', Path.cwd())
print('Branch:')
subprocess.run(['git', 'branch', '--show-current'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'model-training/requirements.txt'], check=True)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

test_data_root = Path('model-training/data/abdul-testrun')
test_run_root = Path('model-training/runs/abdul-testrun')

if test_data_root.exists():
    shutil.rmtree(test_data_root)
if test_run_root.exists():
    shutil.rmtree(test_run_root)

subprocess.run([sys.executable, 'download_abdul_testrun_data.py'], check=True)

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
data_dir = (repo_root / 'model-training' / 'data' / 'abdul-testrun' / 'makeathon-challenge').resolve()
run_dir = (repo_root / 'model-training' / 'runs' / 'abdul-testrun').resolve()

print('Repo root:', repo_root)
print('Data dir:', data_dir)
print('Exists:', data_dir.exists())
print('Run dir:', run_dir)

label_root = data_dir / 'labels' / 'train'
print('Label tif count:', sum(1 for _ in label_root.rglob('*.tif')))

if not data_dir.exists():
    raise FileNotFoundError(f'Dataset not found: {data_dir}')

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        'run_abdul_model.py',
        'all',
        '--data-dir',
        str(data_dir),
        '--run-dir',
        str(run_dir),
    ],
    check=True,
)

In [ ]:
submission = run_dir / 'submission.geojson'
pred_dir = run_dir / 'predictions'
filtered_pred_dir = run_dir / 'filtered_predictions'

print('Submission exists:', submission.exists())
if submission.exists():
    print('Submission:', submission)

print('Prediction files:')
for path in sorted(pred_dir.glob('pred_*.tif'))[:20]:
    print('-', path)

In [ ]:
import json
import geopandas as gpd

with submission.open('r', encoding='utf-8') as f:
    submission_data = json.load(f)

gdf = gpd.GeoDataFrame.from_features(submission_data['features'], crs='EPSG:4326')
print('Anzahl Features:', len(gdf))
display(gdf.head())

In [ ]:
import rasterio
import numpy as np

pred_path = sorted(pred_dir.glob('pred_*.tif'))[0]
with rasterio.open(pred_path) as src:
    arr = src.read(1)

print('Prediction preview:', pred_path.name)
print('shape:', arr.shape)
print('dtype:', arr.dtype)
print('min:', arr.min())
print('max:', arr.max())
print('unique values:', np.unique(arr)[:20])
print('positive pixels:', int((arr > 0).sum()))

In [ ]:
import subprocess
import sys

filtered_pred_dir.mkdir(parents=True, exist_ok=True)
filtered_pred_path = filtered_pred_dir / pred_path.name

subprocess.run(
    [
        sys.executable,
        'apply_prediction_filter.py',
        '--input',
        str(pred_path),
        '--output',
        str(filtered_pred_path),
        '--opening-size',
        '2',
        '--closing-size',
        '2',
        '--min-component-size',
        '64',
    ],
    check=True,
)

with rasterio.open(filtered_pred_path) as src:
    filtered_arr = src.read(1)

print('Filtered preview:', filtered_pred_path.name)
print('shape:', filtered_arr.shape)
print('dtype:', filtered_arr.dtype)
print('min:', filtered_arr.min())
print('max:', filtered_arr.max())
print('unique values:', np.unique(filtered_arr)[:20])
print('positive pixels before:', int((arr > 0).sum()))
print('positive pixels after:', int((filtered_arr > 0).sum()))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(arr, cmap='gray', vmin=0, vmax=max(1, arr.max()))
axes[0].set_title(f'Original: {pred_path.name}')
axes[0].axis('off')

axes[1].imshow(filtered_arr, cmap='gray', vmin=0, vmax=max(1, filtered_arr.max()))
axes[1].set_title(f'Filtered: {filtered_pred_path.name}')
axes[1].axis('off')

plt.tight_layout()
plt.show()